# Imports 

In [1]:
import pandas as pd
from collections import defaultdict
import sys
import os
import shutil as sh
import urllib
import tarfile
import numpy as np
import math
import seaborn as sns
import glob, os
import importlib
import gzip
import MDAnalysis as mda
import nglview as nv
import requests
import json
from biopandas.pdb import PandasPdb
from Bio import AlignIO
import re
from io import StringIO
from sklearn.calibration import LabelEncoder
from sklearn.discriminant_analysis import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial import ConvexHull

from urllib.error import HTTPError
from pathlib import Path
from ipywidgets import interact, interactive, fixed, interact_manual, IntProgress
import ipywidgets as widgets # type: ignore
from IPython.display import display, Markdown, clear_output

#Pandarallel works only on linux and mac
try:
    from pandarallel import pandarallel
    pandarallel.initialize(nb_workers=8,progress_bar=True)
    PARRALEL = True
except:
    PARRALEL = False

from tqdm.notebook import tnrange, tqdm
tqdm.pandas() #activate tqdm progressbar for pandas apply

#Pandas configuration
pd.options.mode.chained_assignment = (
    None  # default='warn', remove pandas warning when adding a new column
)

pd.set_option("display.max_columns", None)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
%config InlineBackend.figure_format ='svg' #better quality figure figure

#%matplotlib inline
sns.set_style("darkgrid")

np.seterr(divide='ignore', invalid='ignore')


INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

# Methods

In [3]:
def load_data(path):
    """
    Load data from the path were it was stored.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"The file at {path} was not found.")
    return pd.read_csv(path)

# Build Dataset Per Superfamily

In [4]:
df = load_data("/home/user_stel/AISB/Project/dataset/processed_dataset.csv")
print(df)

       domain residue_name    IBS  residue_number  convhull_vertex  \
0          PH          ASN  False            19.0              0.0   
1          PH          ASN  False            75.0              1.0   
2          PH          ASN  False            78.0              1.0   
3          PH          ASN  False            92.0              1.0   
4          PH          ASN  False            93.0              1.0   
...       ...          ...    ...             ...              ...   
183885    PLA          CYS  False            91.0              0.0   
183886    PLA          CYS  False            96.0              0.0   
183887    PLA          CYS  False            98.0              0.0   
183888    PLA          CYS  False           105.0              0.0   
183889    PLA          CYS   True           124.0              0.0   

        is_hydrophobic_protrusion  is_co_insertable  exposed  type  
0                             0.0               0.0      1.0   4.0  
1                    

In [ ]:
groups = { domain_name: sub_df 
           for domain_name, sub_df in df.groupby('domain') }

for domain_name, subset_df in groups.items():
    print(f"Domain: {domain_name}  →  Number of rows: {len(subset_df)}")
    display(subset_df)

Domain: ANNEXIN  →  Number of rows: 9283


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
157547,ANNEXIN,ASN,True,55.0,1.0,0.0,0.0,1.0,4.0
157548,ANNEXIN,ASN,False,61.0,1.0,0.0,0.0,1.0,4.0
157549,ANNEXIN,ASN,False,68.0,1.0,0.0,0.0,1.0,4.0
157550,ANNEXIN,ASN,True,17.0,0.0,0.0,0.0,1.0,4.0
157551,ANNEXIN,ASN,True,58.0,1.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
166825,ANNEXIN,CYS,False,270.0,1.0,1.0,0.0,1.0,1.0
166826,ANNEXIN,CYS,False,262.0,1.0,1.0,0.0,1.0,1.0
166827,ANNEXIN,CYS,False,243.0,1.0,1.0,0.0,1.0,1.0
166828,ANNEXIN,CYS,False,242.0,1.0,1.0,0.0,1.0,1.0


Domain: C1  →  Number of rows: 2088


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
79083,C1,ASN,False,156.0,1.0,0.0,0.0,1.0,4.0
79084,C1,ASN,False,154.0,0.0,0.0,0.0,1.0,4.0
79085,C1,ASN,False,175.0,1.0,0.0,0.0,1.0,4.0
79086,C1,ASN,False,199.0,1.0,0.0,0.0,1.0,4.0
79087,C1,ASN,False,204.0,1.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
81166,C1,CYS,False,79.0,0.0,0.0,0.0,0.0,1.0
81167,C1,CYS,False,21.0,1.0,1.0,0.0,1.0,1.0
81168,C1,CYS,True,274.0,0.0,0.0,0.0,1.0,1.0
81169,C1,CYS,True,268.0,0.0,0.0,0.0,1.0,1.0


Domain: C2  →  Number of rows: 15199


...

,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
139085,PLD,ASN,False,319.0,0.0,0.0,0.0,0.0,4.0
139086,PLD,ASN,False,324.0,0.0,0.0,0.0,0.0,4.0
139087,PLD,ASN,False,328.0,0.0,0.0,0.0,0.0,4.0
139088,PLD,ASN,False,409.0,0.0,0.0,0.0,0.0,4.0
139089,PLD,ASN,False,462.0,0.0,0.0,0.0,0.0,4.0
...,...,...,...,...,...,...,...,...,...
157542,PLD,CYS,False,185.0,0.0,0.0,0.0,0.0,1.0
157543,PLD,CYS,False,231.0,0.0,0.0,0.0,0.0,1.0
157544,PLD,CYS,False,233.0,0.0,0.0,0.0,0.0,1.0
157545,PLD,CYS,False,16.0,0.0,0.0,0.0,0.0,1.0


Domain: PX  →  Number of rows: 5847


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
133238,PX,ASN,False,27.0,1.0,0.0,0.0,1.0,4.0
133239,PX,ASN,False,11.0,1.0,0.0,0.0,1.0,4.0
133240,PX,ASN,False,9.0,0.0,0.0,0.0,1.0,4.0
133241,PX,ASN,False,26.0,0.0,0.0,0.0,1.0,4.0
133242,PX,ASN,False,26.0,0.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
139080,PX,CYS,False,138.0,0.0,0.0,0.0,0.0,1.0
139081,PX,CYS,False,133.0,0.0,0.0,0.0,0.0,1.0
139082,PX,CYS,False,133.0,0.0,0.0,0.0,0.0,1.0
139083,PX,CYS,False,41.0,0.0,0.0,0.0,0.0,1.0


Domain: START  →  Number of rows: 32393


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
46690,START,ASN,False,79.0,1.0,0.0,0.0,1.0,4.0
46691,START,ASN,False,95.0,0.0,0.0,0.0,0.0,4.0
46692,START,ASN,True,129.0,0.0,0.0,0.0,0.0,4.0
46693,START,ASN,False,154.0,0.0,0.0,0.0,0.0,4.0
46694,START,ASN,False,15.0,1.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
79078,START,CYS,False,207.0,0.0,0.0,0.0,0.0,1.0
79079,START,CYS,False,207.0,0.0,0.0,0.0,0.0,1.0
79080,START,CYS,False,207.0,0.0,0.0,0.0,0.0,1.0
79081,START,CYS,False,207.0,0.0,0.0,0.0,0.0,1.0


In [7]:
filename = f"{domain_name}_subset.csv"
subset_df.to_csv("/home/user_stel/AISB/Project/dataset/filename", index=False)